# 05 · Anchor products: what is in the first basket of loyal customers?

**Question:** are there products in a customer's **first order** that go together with becoming a **weekly** shopper? If so, the first basket could be used as an early signal during onboarding.

**Loyalty measure:** a customer is *weekly* if their median gap between orders is ≤ 8 days (same rule as notebook 03).

**Caveat:** this analysis finds **correlations, not causes**. Customers who buy a product in their first order may simply be a type of customer who shops weekly anyway.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect("../data/instacart.duckdb", read_only=True)

def q(sql):
    """Run a SQL query on the DuckDB database and return a pandas DataFrame."""
    return con.sql(sql).df()

## 5.1 Baseline: share of weekly customers

In [ ]:
q("""
WITH user_profile AS (
    SELECT user_id,
           CASE WHEN MEDIAN(days_since_prior_order) <= 8 THEN 1 ELSE 0 END AS is_weekly
    FROM orders
    GROUP BY user_id
)
SELECT COUNT(*)                       AS n_users,
       ROUND(AVG(is_weekly) * 100, 1) AS pct_weekly
FROM user_profile
""")

31.9% of customers are weekly shoppers, consistent with notebook 03. This is the baseline for the lift.

## 5.2 What goes into first baskets?

All first orders are in `order_products_prior`: every customer has at least 4 orders and only the last one is `train`/`test`.

In [ ]:
q("""
SELECT p.product_name,
       COUNT(DISTINCT o.user_id)                            AS n_customers,
       ROUND(COUNT(DISTINCT o.user_id) * 100.0 / 206209, 1) AS pct_customers
FROM order_products_prior op
JOIN orders o   ON op.order_id = o.order_id
JOIN products p ON op.product_id = p.product_id
WHERE o.order_number = 1
GROUP BY p.product_id, p.product_name
ORDER BY n_customers DESC
LIMIT 15
""")

First baskets look like regular baskets: the same top sellers as in notebook 04 (bananas, strawberries, avocados, spinach, milk). 14.3% of new customers buy bananas in their first order, so popularity alone cannot identify future loyal customers.

## 5.3 Anchor products

For each product bought in the first order of **at least 1,000 customers** (`HAVING`, to avoid unstable percentages on small groups):

- `pct_weekly`: share of those customers who became weekly shoppers
- `lift` = `pct_weekly` / 31.9 (above 1: more weekly shoppers than average)
- `rank_lift`: rank with the window function `RANK()` (ties share a rank and the next rank is skipped)

### Top 15

In [ ]:
q("""
WITH user_profile AS (
    SELECT user_id,
           CASE WHEN MEDIAN(days_since_prior_order) <= 8 THEN 1 ELSE 0 END AS is_weekly
    FROM orders
    GROUP BY user_id
),
first_items AS (
    SELECT DISTINCT o.user_id, op.product_id
    FROM order_products_prior op
    JOIN orders o ON op.order_id = o.order_id
    WHERE o.order_number = 1
),
anchor AS (
    SELECT p.product_name,
           COUNT(*)                          AS n_customers,
           ROUND(AVG(up.is_weekly) * 100, 1) AS pct_weekly
    FROM first_items f
    JOIN user_profile up ON f.user_id = up.user_id
    JOIN products p      ON f.product_id = p.product_id
    GROUP BY p.product_id, p.product_name
    HAVING COUNT(*) >= 1000
)
SELECT *,
       ROUND(pct_weekly / 31.9, 2)             AS lift,
       RANK() OVER (ORDER BY pct_weekly DESC) AS rank_lift
FROM anchor
ORDER BY rank_lift
LIMIT 15
""")

### Bottom 15

In [ ]:
q("""
WITH user_profile AS (
    SELECT user_id,
           CASE WHEN MEDIAN(days_since_prior_order) <= 8 THEN 1 ELSE 0 END AS is_weekly
    FROM orders
    GROUP BY user_id
),
first_items AS (
    SELECT DISTINCT o.user_id, op.product_id
    FROM order_products_prior op
    JOIN orders o ON op.order_id = o.order_id
    WHERE o.order_number = 1
),
anchor AS (
    SELECT p.product_name,
           COUNT(*)                          AS n_customers,
           ROUND(AVG(up.is_weekly) * 100, 1) AS pct_weekly
    FROM first_items f
    JOIN user_profile up ON f.user_id = up.user_id
    JOIN products p      ON f.product_id = p.product_id
    GROUP BY p.product_id, p.product_name
    HAVING COUNT(*) >= 1000
)
SELECT *,
       ROUND(pct_weekly / 31.9, 2)             AS lift,
       RANK() OVER (ORDER BY pct_weekly ASC) AS rank_lift
FROM anchor
ORDER BY rank_lift
LIMIT 15
""")

- **Top 15:** all fresh, **13 of 15 organic**, with many family/kids items (fruit, milk, string cheese). 40–44% of these customers become weekly shoppers (**lift 1.25–1.37**).
- **Bottom 15:** mostly conventional (**1 of 15 organic**), including many fresh items such as onions, spinach, eggs and tomatoes (**lift 0.50–0.84**). Being fresh alone does not separate loyal customers.
- The organic version of a product tends to rank high and the conventional version low: the signal seems to be the **type of customer** (e.g. families) rather than the product itself.
- Products close to the 1,000-customer threshold (e.g. Sweet Kale Salad Mix, 1,002 customers) are less reliable.

## 5.4 Fresh share of the first basket

Each customer's first order gets a fresh share (% of items from fresh departments). `NTILE(4)` splits customers into 4 equal groups, from the least to the most fresh first basket.

In [ ]:
fresh_q = q("""
WITH user_profile AS (
    SELECT user_id,
           CASE WHEN MEDIAN(days_since_prior_order) <= 8 THEN 1 ELSE 0 END AS is_weekly
    FROM orders
    GROUP BY user_id
),
first_basket AS (
    SELECT o.user_id,
           AVG(CASE WHEN d.department IN ('produce', 'dairy eggs', 'meat seafood', 'bakery', 'deli')
                    THEN 1 ELSE 0 END) * 100 AS fresh_share
    FROM order_products_prior op
    JOIN orders o      ON op.order_id = o.order_id
    JOIN products p    ON op.product_id = p.product_id
    JOIN departments d ON p.department_id = d.department_id
    WHERE o.order_number = 1
    GROUP BY o.user_id
),
quartiles AS (
    SELECT fb.user_id,
           fb.fresh_share,
           up.is_weekly,
           NTILE(4) OVER (ORDER BY fb.fresh_share) AS fresh_quartile
    FROM first_basket fb
    JOIN user_profile up ON fb.user_id = up.user_id
)
SELECT fresh_quartile,
       COUNT(*)                       AS n_users,
       ROUND(AVG(fresh_share), 1)     AS avg_fresh_share,
       ROUND(AVG(is_weekly) * 100, 1) AS pct_weekly
FROM quartiles
GROUP BY fresh_quartile
ORDER BY fresh_quartile
""")
fresh_q

In [ ]:
BASELINE = 31.9  # % weekly customers overall (5.1)

labels = [f"Q{int(qq)}\n{s:.0f}% fresh" for qq, s in zip(fresh_q["fresh_quartile"], fresh_q["avg_fresh_share"])]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, fresh_q["pct_weekly"], color="#2a78d6", width=0.6)
for b, v in zip(bars, fresh_q["pct_weekly"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.8, f"{v:.0f}%", ha="center", fontsize=10, color="#444")
ax.axhline(BASELINE, color="#888", linewidth=1, linestyle="--")
ax.text(len(labels) - 0.5, BASELINE + 0.8, f"all customers: {BASELINE:.0f}%", ha="right", fontsize=9, color="#666")
ax.yaxis.set_visible(False)
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#ccc")
ax.tick_params(axis="x", length=0, labelsize=9)
ax.set_title("Weekly shoppers by fresh share of the first basket", loc="left", fontsize=12)
plt.tight_layout()
plt.savefig("../figures/fresh_first_basket.png", dpi=150, bbox_inches="tight")
plt.show()

The share of weekly shoppers rises steadily from **28.8%** (13% fresh) to **34.7%** (88% fresh): a consistent but moderate effect (~1.2x), weaker than the organic signal in 5.3.

## Findings

1. **First baskets look like regular baskets.** The products most often bought in a first order are the overall top sellers; 14.3% of new customers buy bananas. Popularity alone does not identify future loyal customers.
2. **Organic family items are the strongest signal of a future weekly shopper.** Among products bought in the first order of 1,000+ customers, the top 15 are all fresh and 13 of 15 organic: 40–44% of these customers become weekly shoppers vs 31.9% overall (lift 1.25–1.37). The bottom 15 are mostly conventional, including many fresh items (lift 0.50–0.84).
3. **More fresh food in the first basket goes with more weekly shoppers, but the effect is moderate:** from 28.8% (13% fresh) to 34.7% (88% fresh).

**Recommendation:** the first basket can work as an early signal. A grocer could identify high-potential new customers from their first order (organic fresh items, family products) and give extra onboarding attention (e.g. a second-order incentive or a recurring list) to customers whose first basket suggests a lower chance of becoming regular.

**Caveats:** correlation, not causation (organic items likely reveal a customer type, such as families with higher spending, rather than create loyalty); products near the 1,000-customer threshold are less reliable; no prices or demographics in the data.